## 图像切分为 Patch
假设输入图像：
$$
X\in\mathbb{R}^{H\times W\times C}
$$
其中：
\(H\)：图像高度
\(W\)：图像宽度
\(C\)：通道数，RGB 图像为 3
设 Patch 大小为：
$$
P\times P
$$
Patch 数量：
$$
N=\frac{H}{P}\times\frac{W}{P}
$$
例如：
图像尺寸：224 × 224 × 3
Patch尺寸：16 × 16
则：
$$
N=\frac{224}{16}\times\frac{224}{16}=14\times14=196
$$
因此一张图片被分成 196 个 Patch Token。
## 每个 Patch 变成向量
每个 Patch 的原始形状：
$$
P\times P\times C
$$
展开后维度为：
$$
P^2C
$$
对于 \(16\times16\) 的 RGB Patch：
$$
16\times16\times3=768
$$
每个 Patch 会被展平成：
$$
x_p\in\mathbb{R}^{768}
$$
然后经过线性映射：
$$
z_p=x_pE
$$
其中：
$$
E\in\mathbb{R}^{P^2C\times D}
$$
最终每个 Patch 都被转换为 \(D\) 维 Token。
如果 \(D=768\)，则整个图像变成：
$$
X_{\text{patch}}\in\mathbb{R}^{196\times768}
$$


In [1]:
import torch
import torch.nn as nn


class PatchEmbedding(nn.Module):
    def __init__(
        self,
        image_size: int = 224,
        patch_size: int = 16,
        in_channels: int = 3,
        embed_dim: int = 768,
    ):
        super().__init__()

        self.num_patches = (image_size // patch_size) ** 2

        self.projection = nn.Conv2d(
            in_channels=in_channels,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        # images: [B, C, H, W] -> [B, D, H/P, W/P]
        x = self.projection(images)

        # [B, D, H/P, W/P] -> [B, D, H/P*W/P] = [B, D, N]
        x = x.flatten(2)

        # [B, D, N] -> [B, N, D]
        x = x.transpose(1, 2)

        return x


images = torch.randn(4, 3, 224, 224)

patch_embedding = PatchEmbedding()
tokens = patch_embedding(images)

print(tokens.shape)

torch.Size([4, 196, 768])
